# Article 1 — Avellaneda–Stoikov Market Making

**Classical diffusions, inventory control, and covariance-aware extension**

Authors:

- Salvatore Gabriele Messina — https://www.linkedin.com/in/salvatore-messinaa/
- Niccolò Soriano — https://www.linkedin.com/in/niccol%C3%B2-soriano/

This notebook is a GitHub-oriented reading companion. It does not replace the executable simulation notebook `articolo1_applicazione.ipynb`; it summarises the model, records the main formulas, and includes a complete multivariate application example that runs end-to-end.

## 1. Article Map

Article 1 builds a clean benchmark for high-frequency market making under the Avellaneda–Stoikov idea:

- define the limit order book and reduced-form fill process;
- compare naive symmetric quoting with inventory-aware quoting;
- run ABM and GBM mid-price dynamics with the same execution/accounting engine;
- extend to a covariance-aware multi-asset setting where scalar inventory risk becomes portfolio covariance risk.

## 2. Single-Asset Core

The baseline arithmetic mid-price model is

$$
dS_t = \sigma\,dW_t.
$$

The dealer maximises expected exponential (CARA) utility of terminal wealth

$$
\mathbb{E}\bigl[-e^{-\gamma W_T}\bigr], \quad W_T = X_T + q_T S_T,
$$

where $X_T$ is cash, $q_T$ is terminal inventory, and $\gamma>0$ is risk aversion.

The HJB approach (Avellaneda–Stoikov 2008) yields the **reservation price**

$$
r_t = S_t - q_t\,\gamma\,\sigma^2\,(T-t),
$$

which shifts quotes toward the side that reduces $q_t$.
For exponential fill intensity $\lambda(\delta)=A\,e^{-k\delta}$,
the optimal liquidity half-spread is constant:

$$
\delta^* = \frac{1}{\gamma}\ln\!\left(1+\frac{\gamma}{k}\right).
$$

The **AS bid/ask quotes** are therefore

$$
p^b_t = r_t - \delta^*, \qquad p^a_t = r_t + \delta^*.
$$

In [ ]:
import numpy as np

P = dict(S0=100.0, T=1.0, sigma_BM=1.5, gamma=0.1, k=1.5, A=140.0)
delta_star = (1.0 / P['gamma']) * np.log(1.0 + P['gamma'] / P['k'])
print(f"delta* = {delta_star:.6f}")

def reservation_price_single(S, q, t, gamma, sigma, T):
    tau = max(T - t, 0.0)
    return S - q * gamma * sigma**2 * tau

def quotes_as_single(S, q, t, gamma, k, sigma, T):
    r = reservation_price_single(S, q, t, gamma, sigma, T)
    delta = (1.0 / gamma) * np.log(1.0 + gamma / k)
    return r - delta, r + delta

for q in [-5, 0, 5]:
    bid, ask = quotes_as_single(100.0, q, 0.25, P['gamma'], P['k'], P['sigma_BM'], P['T'])
    print(f"q={q:+d}: bid={bid:.4f}, ask={ask:.4f}")

## 3. Why GBM Changes the Risk Scale

Under geometric Brownian motion,

$$
dS_t = \sigma_{\mathrm{GBM}}\,S_t\,dW_t,
$$

the **dollar volatility** of one share is state-dependent: $\sigma_{\mathrm{GBM}}\,S_t$.
To match dollar-scale risk with the ABM case we set $\sigma_{\mathrm{GBM}}\cdot S_0 = \sigma_{\mathrm{BM}}$.

The heuristic reservation price used in the simulation is

$$
r_t^{\mathrm{GBM}} = S_t - q_t\,\gamma\,(\sigma_{\mathrm{GBM}}\,S_t)^2\,(T-t),
$$

so the inventory penalty grows with $S_t^2$: a high-price regime magnifies inventory risk.

In [ ]:
def quotes_as_gbm(S, q, t, gamma, k, sigma_gbm, T):
    tau = max(T - t, 0.0)
    sigma_eff = sigma_gbm * S
    r = S - q * gamma * sigma_eff**2 * tau
    delta = (1.0 / gamma) * np.log(1.0 + gamma / k)
    return r - delta, r + delta

for S in [80.0, 100.0, 120.0]:
    bid, ask = quotes_as_gbm(S, q=5, t=0.25, gamma=0.1, k=1.5, sigma_gbm=0.015, T=1.0)
    print(f"S={S:6.2f}: bid={bid:.4f}, ask={ask:.4f}")

## 4. Multivariate Extension

For $d$ assets, prices and inventories become vectors:

$$
S_t\in\mathbb{R}^d, \qquad q_t\in\mathbb{Z}^d, \qquad X_t\in\mathbb{R}.
$$

Correlated arithmetic Brownian motions are generated via the Cholesky factor $C$
of the instantaneous covariance matrix $\Sigma = CC^\top$:

$$
dS_t = C\,d\widetilde{W}_t, \qquad \widetilde{W}_t\in\mathbb{R}^d\text{ i.i.d. standard BM.}
$$

Mark-to-market wealth is

$$
W_t = X_t + q_t^\top S_t.
$$

The frozen-inventory portfolio variance over the remaining horizon $\tau = T-t$ is

$$
\mathrm{Var}_t\bigl(q_t^\top(S_T - S_t)\bigr) = \tau\,q_t^\top\Sigma\,q_t.
$$

The gradient of the quadratic risk charge with respect to $q$ is the **covariance pressure**:

$$
\nabla_q\!\left(\frac{\gamma}{2}\,\tau\,q_t^\top\Sigma\,q_t\right) = \gamma\,\tau\,\Sigma\,q_t.
$$

This is the key object: quote $i$ reacts to $(\Sigma q_t)_i$, not just $q_t^i$.
A long position in correlated asset $j$ pushes the dealer to tighten the ask of asset $i$.

In [ ]:
Sigma = np.array([[2.25,   1.0725, -0.4725],
                  [1.0725, 1.21,    0.2475],
                  [-0.4725, 0.2475,  0.81]])
q = np.array([8, 0, -5])
pressure = Sigma @ q
risk_norm = np.sqrt(q @ Sigma @ q)
print('q           =', q)
print('Sigma @ q   =', pressure)
print(f'risk norm   = {risk_norm:.4f}')

## 5. Marginal Multi-Asset Reservation Prices

Buying one extra unit of asset $i$ shifts portfolio inventory from $q$ to $q + e_i$.
The marginal change in risk charge is $\gamma\tau(\Sigma q)_i + \tfrac{\gamma}{2}\tau\Sigma_{ii}$.
Splitting bid and ask symmetrically around the covariance-adjusted center gives

$$
r_{i}^{\mathrm{bid}} = S_i - \gamma\tau(\Sigma q)_i - \tfrac{\gamma}{2}\tau\Sigma_{ii},
$$

$$
r_{i}^{\mathrm{ask}} = S_i - \gamma\tau(\Sigma q)_i + \tfrac{\gamma}{2}\tau\Sigma_{ii}.
$$

The final executable quotes add/subtract the per-asset liquidity half-spread $\delta_i^*$:

$$
p_i^b = r_i^{\mathrm{bid}} - \delta_i^*, \qquad p_i^a = r_i^{\mathrm{ask}} + \delta_i^*,
\qquad \delta_i^* = \frac{1}{\gamma}\ln\!\left(1+\frac{\gamma}{k_i}\right).
$$

In [ ]:
def quotes_as_multi(S, q, t, gamma, k, Sigma, T):
    S = np.asarray(S, dtype=float)
    q = np.asarray(q, dtype=float)
    k = np.asarray(k, dtype=float)
    tau = max(T - t, 0.0)
    pressure = Sigma @ q
    risk_half = 0.5 * gamma * tau * np.diag(Sigma)
    liq_half = (1.0 / gamma) * np.log(1.0 + gamma / k)
    bid_res = S - gamma * tau * pressure - risk_half
    ask_res = S - gamma * tau * pressure + risk_half
    return bid_res - liq_half, ask_res + liq_half

S0 = np.array([100.0, 75.0, 50.0])
k = np.array([1.50, 1.80, 1.65])
bid, ask = quotes_as_multi(S0, q, t=0.0, gamma=0.1, k=k, Sigma=Sigma, T=1.0)
print('bid =', np.round(bid, 4))
print('ask =', np.round(ask, 4))

## 6. Multivariate Simulation — Infrastructure

This section implements the full correlated multi-asset simulation engine.
Fill probabilities use the same exponential intensity model as the single-asset case,
vectorised per asset:

$$
\lambda_i^b(\delta_i^b) = A_i\,e^{-k_i\delta_i^b},
\qquad
P_{\mathrm{fill},i}^b = 1 - e^{-\lambda_i^b\,\Delta t}.
$$

Each asset's cash flow and inventory evolve independently conditioned on the shared
price path; total mark-to-market PnL is $W_t = X_t + q_t^\top S_t$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'font.family': 'serif', 'font.size': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3, 'figure.dpi': 110,
})


def sim_multi_BM(dW, S0, chol_cov):
    '''Correlated arithmetic Brownian motion: dS = C dW_tilde.'''
    n, d = dW.shape
    S = np.empty((n + 1, d))
    S[0] = np.asarray(S0, dtype=float)
    dS = dW @ chol_cov.T
    for i in range(n):
        S[i+1] = S[i] + dS[i]
    return S


def portfolio_risk_norm(q, Sigma):
    '''sqrt(q^T Sigma q) — instantaneous portfolio risk.'''
    q = np.asarray(q, dtype=float)
    return float(np.sqrt(max(q @ Sigma @ q, 0.0)))


def covariance_pressure(q, Sigma):
    '''Sigma @ q — gradient direction of the quadratic inventory penalty.'''
    return Sigma @ np.asarray(q, dtype=float)


def fill_probs_multi(S, p_bid, p_ask, A, k, dt):
    '''Vectorised per-asset fill probabilities.'''
    S = np.asarray(S); p_bid = np.asarray(p_bid); p_ask = np.asarray(p_ask)
    A = np.asarray(A); k = np.asarray(k)
    lam_b = A * np.exp(-k * np.maximum(S - p_bid, 0.0))
    lam_a = A * np.exp(-k * np.maximum(p_ask - S, 0.0))
    return 1.0 - np.exp(-lam_b * dt), 1.0 - np.exp(-lam_a * dt)


def quote_naive_multi(S, q, t, gamma, k, **_):
    '''Independent symmetric quotes: p^b/a = S -/+ delta*.'''
    S = np.asarray(S, dtype=float)
    delta = (1.0 / gamma) * np.log(1.0 + gamma / np.asarray(k))
    return S - delta, S + delta


def quote_as_multi_cov(S, q, t, gamma, k, Sigma, T, **_):
    '''
    Covariance-aware AS quotes.
    Center = S - gamma*tau*(Sigma@q); bid/ask split adds +/- 0.5*gamma*tau*diag(Sigma).
    '''
    S = np.asarray(S, dtype=float)
    q = np.asarray(q, dtype=float)
    k = np.asarray(k, dtype=float)
    tau = max(T - t, 0.0)
    pressure = Sigma @ q
    risk_half = 0.5 * gamma * tau * np.diag(Sigma)
    liq_half = (1.0 / gamma) * np.log(1.0 + gamma / k)
    bid_res = S - gamma * tau * pressure - risk_half
    ask_res = S - gamma * tau * pressure + risk_half
    return bid_res - liq_half, ask_res + liq_half


def run_sim_multi(t_arr, S, U_bid, U_ask, qfn, dt, **kw):
    '''Portfolio accounting for a vector of assets.'''
    A = kw['A']; k = kw['k']
    n, d = S.shape[0] - 1, S.shape[1]
    inv  = np.zeros((n + 1, d), dtype=int)
    cash = np.zeros(n + 1)
    pnl  = np.zeros(n + 1)
    risk = np.zeros(n + 1)
    bids = np.zeros((n + 1, d))
    asks = np.zeros((n + 1, d))
    q = np.zeros(d, dtype=int)
    c = 0.0
    Sigma_kw = kw.get('Sigma')
    for i in range(n):
        s = S[i]; t = t_arr[i]
        pb, pa = qfn(s, q, t, **kw)
        bids[i] = pb; asks[i] = pa
        pfb, pfa = fill_probs_multi(s, pb, pa, A, k, dt)
        bid_fill = U_bid[i] < pfb
        ask_fill = U_ask[i] < pfa
        q  += bid_fill.astype(int)
        c  -= float(np.sum(pb * bid_fill))
        q  -= ask_fill.astype(int)
        c  += float(np.sum(pa * ask_fill))
        inv[i+1]  = q
        cash[i+1] = c
        pnl[i+1]  = c + float(q @ S[i+1])
        if Sigma_kw is not None:
            risk[i+1] = portfolio_risk_norm(q, Sigma_kw)
    pb, pa = qfn(S[-1], q, t_arr[-1], **kw)
    bids[-1] = pb; asks[-1] = pa
    return dict(inv=inv, cash=cash, pnl=pnl, risk=risk, bids=bids, asks=asks)


def summarize_multi(res, Sigma):
    qT = res['inv'][-1].astype(float)
    return dict(
        pnl          = float(res['pnl'][-1]),
        inv          = qT,
        gross_abs_inv= float(np.sum(np.abs(qT))),
        risk_norm    = portfolio_risk_norm(qT, Sigma),
        pressure     = covariance_pressure(qT, Sigma),
    )

print('Simulation infrastructure loaded.')

## 7. Three-Asset Application Example

We simulate three correlated assets on the horizon $T=1$ with $n=1000$ steps.
The instantaneous covariance matrix is

$$
\Sigma = \begin{pmatrix}
  \sigma_A^2 & \rho_{AB}\sigma_A\sigma_B & \rho_{AC}\sigma_A\sigma_C \\
  \rho_{AB}\sigma_A\sigma_B & \sigma_B^2 & \rho_{BC}\sigma_B\sigma_C \\
  \rho_{AC}\sigma_A\sigma_C & \rho_{BC}\sigma_B\sigma_C & \sigma_C^2
\end{pmatrix},
$$

with $\sigma_A=1.5$, $\sigma_B=1.1$, $\sigma_C=0.9$ and correlations
$\rho_{AB}=0.65$, $\rho_{AC}=-0.35$, $\rho_{BC}=0.25$.

The Cholesky factor $C$ (lower triangular, $\Sigma = CC^\top$) drives the correlated increments.
We compare **naive independent symmetric quoting** against
**covariance-aware AS quoting** (AS-cov) on the same random path.

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────
T       = 1.0
n_steps = 1000
dt      = T / n_steps
gamma   = 0.1
seed    = 42

names   = ['Asset A', 'Asset B', 'Asset C']
S0      = np.array([100.0,  75.0,  50.0])
sigma   = np.array([  1.50,   1.10,   0.90])
corr    = np.array([[1.00,  0.65, -0.35],
                    [0.65,  1.00,  0.25],
                    [-0.35, 0.25,  1.00]])
A_arr   = np.array([140.0, 120.0, 100.0])
k_arr   = np.array([  1.50,   1.80,   1.65])

Sigma_3 = np.outer(sigma, sigma) * corr
chol_3  = np.linalg.cholesky(Sigma_3)

print('Covariance matrix Sigma:')
print(np.round(Sigma_3, 4))
print()
print('Cholesky factor C  (Sigma = C C^T):')
print(np.round(chol_3, 4))

In [ ]:
# ── Single-path simulation ─────────────────────────────────────────────────
rng   = np.random.default_rng(seed)
t_arr = np.linspace(0, T, n_steps + 1)
dW    = rng.normal(0, np.sqrt(dt), (n_steps, 3))
Ub    = rng.random((n_steps, 3))
Ua    = rng.random((n_steps, 3))

S_3 = sim_multi_BM(dW, S0, chol_3)

kw_naive_3 = dict(gamma=gamma, k=k_arr, A=A_arr)
kw_as_3    = dict(gamma=gamma, k=k_arr, A=A_arr, Sigma=Sigma_3, T=T)

r_naive_3 = run_sim_multi(t_arr, S_3, Ub, Ua, quote_naive_multi,   dt, **kw_naive_3)
r_as_3    = run_sim_multi(t_arr, S_3, Ub, Ua, quote_as_multi_cov,  dt, **kw_as_3)

sn = summarize_multi(r_naive_3, Sigma_3)
sa = summarize_multi(r_as_3,    Sigma_3)

for label, out in [('Naive', sn), ('AS-cov', sa)]:
    print(f'{label:<8}: PnL={out["pnl"]:+9.3f}  |q_T|={out["gross_abs_inv"]:5.1f}  risk={out["risk_norm"]:.4f}')
    print(f'         inv      = {out["inv"]}')
    print(f'         Sigma@q  = {np.round(out["pressure"], 4)}')

# Demo: covariance pressure for a specific inventory
q_demo = np.array([8, 0, -5])
pb_demo, pa_demo = quote_as_multi_cov(
    S0, q_demo, 0.0, gamma=gamma, k=k_arr, Sigma=Sigma_3, T=T, A=A_arr
)
print()
print('Demo inventory q =', q_demo)
print('Sigma @ q        =', np.round(covariance_pressure(q_demo, Sigma_3), 4))
print('AS-cov bid       =', np.round(pb_demo, 4))
print('AS-cov ask       =', np.round(pa_demo, 4))

## 8. Multivariate Monte Carlo Diagnostic

We run $N=500$ independent Monte Carlo paths (reduced from 5,000 for interactive use)
and compare the terminal distribution of

$$
W_T = X_T + q_T^\top S_T,
\qquad
\|q_T\|_1 = \sum_{i=1}^d |q_T^i|,
\qquad
\|q_T\|_\Sigma = \sqrt{q_T^\top\Sigma\,q_T}.
$$

The last metric is the instantaneous portfolio risk at the terminal inventory.

In [ ]:
N_MC = 500
master = np.random.default_rng(3000)
dW_all = master.normal(0, np.sqrt(dt), (N_MC, n_steps, 3))
Ub_all = master.random((N_MC, n_steps, 3))
Ua_all = master.random((N_MC, n_steps, 3))

pnl_n  = np.zeros(N_MC); pnl_as  = np.zeros(N_MC)
gabs_n = np.zeros(N_MC); gabs_as = np.zeros(N_MC)
risk_n = np.zeros(N_MC); risk_as = np.zeros(N_MC)

for m in range(N_MC):
    S_m  = sim_multi_BM(dW_all[m], S0, chol_3)
    rn   = run_sim_multi(t_arr, S_m, Ub_all[m], Ua_all[m],
                         quote_naive_multi,  dt, **kw_naive_3)
    ra   = run_sim_multi(t_arr, S_m, Ub_all[m], Ua_all[m],
                         quote_as_multi_cov, dt, **kw_as_3)
    sn_m = summarize_multi(rn, Sigma_3)
    sa_m = summarize_multi(ra, Sigma_3)
    pnl_n[m]  = sn_m['pnl'];  pnl_as[m]  = sa_m['pnl']
    gabs_n[m] = sn_m['gross_abs_inv']; gabs_as[m] = sa_m['gross_abs_inv']
    risk_n[m] = sn_m['risk_norm'];     risk_as[m] = sa_m['risk_norm']

print(f'{"Metric":<30} {"Naive":>10} {"AS-cov":>10}')
print('-' * 52)
for label, a, b in [
    ('Mean terminal PnL',   np.mean(pnl_n),  np.mean(pnl_as)),
    ('Std  terminal PnL',   np.std(pnl_n),   np.std(pnl_as)),
    ('Sharpe-like ratio',
     np.mean(pnl_n)/(np.std(pnl_n)+1e-12),
     np.mean(pnl_as)/(np.std(pnl_as)+1e-12)),
    ('Mean |q_T| (L1)',     np.mean(gabs_n), np.mean(gabs_as)),
    ('Mean risk norm ||q||_Sigma', np.mean(risk_n), np.mean(risk_as)),
]:
    print(f'{label:<30} {a:>10.3f} {b:>10.3f}')

## 9. Portfolio Risk Visualization

The 2×2 figure below shows, for the single reference path:

- **(a)** mid-price paths for all three assets;
- **(b)** portfolio PnL $W_t = X_t + q_t^\top S_t$ for naive vs AS-cov;
- **(c)** portfolio risk norm $\|q_t\|_\Sigma$ over time;
- **(d)** terminal PnL histogram over all $N$ Monte Carlo paths.

In [ ]:
COLORS = dict(naive='#4878CF', AS='#C44E52')
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle('Covariance-Aware Multi-Asset Market Making — 3 Correlated ABM Assets',
             fontsize=12, fontweight='bold')

# (a) mid-price paths
ax = axes[0, 0]
for i, nm in enumerate(names):
    ax.plot(t_arr, S_3[:, i], lw=1.0, label=nm)
ax.set_xlabel('Time $t$'); ax.set_ylabel('Price $S_t^i$')
ax.set_title('(a) Correlated Mid-Price Paths'); ax.legend(fontsize=9)

# (b) portfolio PnL
ax = axes[0, 1]
ax.plot(t_arr, r_naive_3['pnl'], color=COLORS['naive'], lw=1.1, label='Naive')
ax.plot(t_arr, r_as_3['pnl'],   color=COLORS['AS'],    lw=1.1, ls='--', label='AS-cov')
ax.axhline(0, color='gray', lw=0.7, ls=':', alpha=0.6)
ax.set_xlabel('Time $t$'); ax.set_ylabel(r'PnL $W_t = X_t + q_t^\top S_t$')
ax.set_title('(b) Portfolio Mark-to-Market PnL'); ax.legend(fontsize=9)

# (c) portfolio risk norm
ax = axes[1, 0]
ax.plot(t_arr, r_naive_3['risk'], color=COLORS['naive'], lw=1.0, label='Naive')
ax.plot(t_arr, r_as_3['risk'],   color=COLORS['AS'],    lw=1.0, ls='--', label='AS-cov')
ax.set_xlabel('Time $t$'); ax.set_ylabel(r'$\|q_t\|_\Sigma = \sqrt{q_t^\top \Sigma q_t}$')
ax.set_title(r'(c) Portfolio Risk Norm $\|q_t\|_\Sigma$'); ax.legend(fontsize=9)

# (d) terminal PnL histogram
ax = axes[1, 1]
all_v = np.concatenate([pnl_n, pnl_as])
lo, hi = np.percentile(all_v, 1), np.percentile(all_v, 99)
bins = np.linspace(lo, hi, 55)
ax.hist(pnl_n,  bins=bins, color=COLORS['naive'], alpha=0.50, density=True, label='Naive')
ax.hist(pnl_as, bins=bins, color=COLORS['AS'],    alpha=0.50, density=True, label='AS-cov')
ax.axvline(np.mean(pnl_n),  color=COLORS['naive'], lw=1.5, ls='--')
ax.axvline(np.mean(pnl_as), color=COLORS['AS'],    lw=1.5, ls='--')
ax.set_xlabel('Terminal PnL $W_T$'); ax.set_ylabel('Density')
ax.set_title(f'(d) Terminal PnL ({N_MC} MC paths)'); ax.legend(fontsize=9)

fig.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

## 10. Reproducing the Full Simulation

The full executable notebook is generated from `_gen_notebook.py` and writes:

- `articolo1_applicazione.ipynb`
- `py1.png` for the ABM comparison
- `py2.png` for the GBM comparison

Run these commands locally from the article folder when notebook dependencies are available:

```powershell
& "C:\\Users\\salvm\\AppData\\Roaming\\jupyterlab-desktop\\jlab_server\\python.exe" _gen_notebook.py
& "C:\\Users\\salvm\\AppData\\Roaming\\jupyterlab-desktop\\jlab_server\\python.exe" -m jupyter nbconvert --execute --inplace articolo1_applicazione.ipynb
```

In [ ]:
from pathlib import Path
from IPython.display import Image, display

for name in ['py1.png', 'py2.png']:
    path = Path(name)
    if path.exists():
        print(f'{name}: {path.stat().st_size:,} bytes')
        display(Image(filename=str(path)))
    else:
        print(f'{name}: missing — run articolo1_applicazione.ipynb first')

## 11. References

- Avellaneda and Stoikov (2008), *High-frequency trading in a limit order book*.
- Cartea, Jaimungal, and Penalva (2015), *Algorithmic and High-Frequency Trading*.
- Guéant (2016), *The Financial Mathematics of Market Liquidity: From Optimal Execution to Market Making*.
- Bergault, Evangelista, Guéant, and Vieira (2022), *Closed-form approximations in multi-asset market making*.